In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

def get_app_ids(num_pages=300):
    app_ids = []

    for page in range(1, num_pages + 1):
        url = f"https://store.steampowered.com/search/?category1=998&page={page}"
        
        try:
            res = requests.get(url, headers=HEADERS, timeout=10)
            soup = BeautifulSoup(res.text, 'html.parser')

            for link in soup.find_all('a', {'data-ds-appid': True}):
                app_ids.append(link['data-ds-appid'])

            print(f"Collected IDs from page {page}")
            time.sleep(1)

        except Exception as e:
            print(f"Error on page {page}: {e}")

    return list(set(app_ids)) 

def get_game_details(app_id):
    api_url = f"https://store.steampowered.com/api/appdetails?appids={app_id}"

    try:
        res = requests.get(api_url, headers=HEADERS, timeout=10).json()

        if res and str(app_id) in res and res[str(app_id)]['success']:
            data = res[str(app_id)]['data']

        
            price_info = data.get("price_overview", {})
            initial_price = price_info.get("initial", 0) / 100.0
            final_price = price_info.get("final", 0) / 100.0
            discount_pct = price_info.get("discount_percent", 0)

            review_count = data.get("recommendations", {}).get("total", 0)

            genres = [g['description'] for g in data.get("genres", [])]
            categories = [c['description'] for c in data.get("categories", [])]
            all_tags = ", ".join(genres + categories)

            metascore = data.get("metacritic", {}).get("score")

            return {
                "Title": data.get("name"),
                "Original Price": initial_price,
                "Current Price": final_price,
                "Discount %": discount_pct,
                "Review Count": review_count,
                "Tags": all_tags,
                "Platform": ", ".join([k for k, v in data.get("platforms", {}).items() if v]),
                "Release Date": data.get("release_date", {}).get("date"),
                "Metascore": metascore,
                "Developer": ", ".join(data.get("developers", [])),
                "Publisher": ", ".join(data.get("publishers", [])),
                "Summary": data.get("short_description")
            }

    except Exception as e:
        print(f"Error on AppID {app_id}: {e}")

    return None

all_ids = get_app_ids(num_pages=500)

game_list = []

for i, app_id in enumerate(all_ids):
    details = get_game_details(app_id)

    if details:
        game_list.append(details)

    if i % 20 == 0:
        print(f"Scraped {i}/{len(all_ids)} games...")

    time.sleep(1.2)

df = pd.DataFrame(game_list)

print(f"Total games collected: {len(df)}")

df.to_csv("Games_dataset_full.csv", index=False)

df_with_score = df[df["Metascore"].notnull()]
df_with_score.to_csv("Games_with_metascore.csv", index=False)

df_discounted = df[df["Discount %"] > 0]
df_discounted.to_csv("Games_discounted.csv", index=False)

print("All datasets created successfully!")

Collected IDs from page 1
Collected IDs from page 2
Collected IDs from page 3
Collected IDs from page 4
Collected IDs from page 5
Collected IDs from page 6
Collected IDs from page 7
Collected IDs from page 8
Collected IDs from page 9
Collected IDs from page 10
Collected IDs from page 11
Collected IDs from page 12
Collected IDs from page 13
Collected IDs from page 14
Collected IDs from page 15
Collected IDs from page 16
Collected IDs from page 17
Collected IDs from page 18
Collected IDs from page 19
Collected IDs from page 20
Collected IDs from page 21
Collected IDs from page 22
Collected IDs from page 23
Collected IDs from page 24
Collected IDs from page 25
Collected IDs from page 26
Collected IDs from page 27
Collected IDs from page 28
Collected IDs from page 29
Collected IDs from page 30
Collected IDs from page 31
Collected IDs from page 32
Collected IDs from page 33
Collected IDs from page 34
Collected IDs from page 35
Collected IDs from page 36
Collected IDs from page 37
Collected 

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9780 entries, 0 to 9779
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Title           9780 non-null   object 
 1   Original Price  9780 non-null   float64
 2   Current Price   9780 non-null   float64
 3   Discount %      9780 non-null   int64  
 4   Review Count    9780 non-null   int64  
 5   Tags            9780 non-null   object 
 6   Platform        9780 non-null   object 
 7   Release Date    9780 non-null   object 
 8   Metascore       1799 non-null   float64
 9   Developer       9780 non-null   object 
 10  Publisher       9780 non-null   object 
 11  Summary         9780 non-null   object 
dtypes: float64(3), int64(2), object(7)
memory usage: 917.0+ KB


In [4]:
df_with_score.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1799 entries, 4 to 9779
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Title           1799 non-null   object 
 1   Original Price  1799 non-null   float64
 2   Current Price   1799 non-null   float64
 3   Discount %      1799 non-null   int64  
 4   Review Count    1799 non-null   int64  
 5   Tags            1799 non-null   object 
 6   Platform        1799 non-null   object 
 7   Release Date    1799 non-null   object 
 8   Metascore       1799 non-null   float64
 9   Developer       1799 non-null   object 
 10  Publisher       1799 non-null   object 
 11  Summary         1799 non-null   object 
dtypes: float64(3), int64(2), object(7)
memory usage: 182.7+ KB


In [6]:
df_discounted.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1856 entries, 4 to 9779
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Title           1856 non-null   object 
 1   Original Price  1856 non-null   float64
 2   Current Price   1856 non-null   float64
 3   Discount %      1856 non-null   int64  
 4   Review Count    1856 non-null   int64  
 5   Tags            1856 non-null   object 
 6   Platform        1856 non-null   object 
 7   Release Date    1856 non-null   object 
 8   Metascore       377 non-null    float64
 9   Developer       1856 non-null   object 
 10  Publisher       1856 non-null   object 
 11  Summary         1856 non-null   object 
dtypes: float64(3), int64(2), object(7)
memory usage: 188.5+ KB
